# HW2: Policy Gradiant
> - Full Name: **REDACTED**
> - Student ID: **REDACTED**


## Objective

In this notebook, you will implement the **REINFORCE policy gradient algorithm** (also known as Monte Carlo policy gradient) to train an agent on the **CartPole‑v1** environment from OpenAI Gym. REINFORCE is a fundamental policy‑based reinforcement learning method that directly optimises the policy parameters by performing stochastic gradient ascent on the expected cumulative reward. However, the standard REINFORCE update is known to suffer from **high variance**, which can lead to slow convergence and unstable learning curves.

## Variance Reduction Techniques

To address this issue, this notebook further explores three variance‑reduction strategies that are commonly applied to the REINFORCE algorithm. You will implement each of them and empirically analyse their effects on learning speed, stability, and final performance. The techniques are:

1. **The causality trick (reward‑to‑go)**  
   Instead of using the full return from the start of an episode, this method replaces the cumulative reward with the sum of future rewards from each time step onward. By ignoring rewards obtained before the current action, the gradient estimator retains the same unbiasedness while significantly reducing variance.

2. **Constant baseline**  
   A fixed scalar baseline is subtracted from the return (or the reward‑to‑go) in the policy gradient estimate. Subtracting a constant does not introduce bias, but a well‑chosen constant (e.g., the average return over recent episodes) can reduce variance by centring the advantage estimates.

3. **Learned baseline (value function baseline)**  
   A more sophisticated approach that uses a separate function approximator (typically a neural network) to estimate the state‑value function. This learned baseline adapts during training and provides a state‑dependent offset. The advantage estimate then often yields the lowest variance among these three methods, at the cost of adding a second learning process.

## Implementation and Analysis

You will implement the core REINFORCE update, then systematically extend it with each of the three variance‑reduction techniques. For every variant, you will run multiple training episodes, plot the smoothed episode returns, and compare convergence behaviour. The notebook will guide you through:

- Setting up the CartPole‑v1 environment (discrete action space, continuous observation space).
- Defining a policy network (e.g., a shallow multi‑layer perceptron) with a softmax output layer.
- Writing the training loop that collects episodes, computes policy gradients, and updates the policy.
- Modifying the advantage estimator to incorporate causality and baseline subtractions.
- Training and evaluating each variant with identical random seeds for fair comparison.

By the end of this notebook, you will gain hands‑on insight into how different variance‑reduction mechanisms affect the practical performance of Monte Carlo policy gradient methods.

# Setup

In [ ]:
%pip install -q "gymnasium[classic-control]"

In [ ]:
%pip install -q imageio-ffmpeg

All required packages are pre-installed if using Google Colab.


Import the following libraries.

In [ ]:
# Imports
import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

import gymnasium as gym
import random
import matplotlib
import matplotlib.pyplot as plt
import base64
import imageio
import IPython
import logging
import warnings
import imageio.v2 as imageio
from IPython.display import display


# Disable warnings
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# DEVICE
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

Note: While we check for CUDA availability, using a GPU is not strictly necessary for this assignment. The neural networks used here are small, and the majority of training time is spent running episodes in the environment (which occurs on the CPU), rather than performing backpropagation updates.

Set a fix seed to be able to compare results

In [ ]:
# We will use set_seed later.
def set_seed(seed=333):
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Configure Matplotlib for Interactive and XKCD-Style Plots

In [ ]:
# Set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython.display import display

plt.ion()
plt.xkcd(scale=1, length=100, randomness=2)
matplotlib.rcParams['figure.figsize'] = (12, 6)

Record and Embed Simulation Videos in Jupyter Notebook

<b>embed_mp4:</b> Converts an MP4 video into a base64-encoded HTML tag for display in Jupyter Notebook.
<br>
<b>record_simulation:</b> Runs a policy in the environment, records the simulation, and saves it as an MP4 video.

In [ ]:
def embed_mp4(filename):
    video = open(filename,'rb').read()
    b64 = base64.b64encode(video)
    tag = '''
    <video width="640" height="480" controls>
    <source src="data:video/mp4;base64,{0}" type="video/mp4">
    Your browser does not support the video tag.
    </video>'''.format(b64.decode())
    return IPython.display.HTML(tag)

def record_simulation(env, policy_net, filename, episodes=1, fps=30):
    filename = filename + ".mp4"
    frames = []

    for _ in range(episodes):
        state, info = env.reset()

        frame = env.render()
        frames.append(np.asarray(frame, dtype=np.uint8))

        done = False
        while not done:
            state_tensor = torch.FloatTensor(state).to(DEVICE)

            with torch.no_grad():
                action = torch.argmax(policy_net(state_tensor)).item()

            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            frame = env.render()
            frames.append(np.asarray(frame, dtype=np.uint8))

    env.close()

    imageio.mimsave(
        filename,
        frames,
        fps=fps,
        format="FFMPEG",
        codec="libx264",
        macro_block_size=16
    )

    return embed_mp4(filename)

# Explore the environment

Initialize CartPole Environment and Display State & Action Spaces

In [ ]:
# Create the CartPole environment
env = gym.make("CartPole-v1")

# Print observation and action space
print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)

**Question**

How are the observation and action spaces defined in the CartPole environment?

**Answer:**

The CartPole observation space is a continuous 4-dimensional vector representing the cart position, cart velocity, pole angle, and pole angular velocity, and action space is discrete with 2 possible actions, pushing the cart left or right.

Define a Random Policy for Action Selection  

In [ ]:
class RandomPolicy(object):

    def __init__(self, action_space_n):
        self.action_space_n = action_space_n

    def __call__(self, state):
        action_values = torch.rand(self.action_space_n)
        return action_values

Visualize the random policy.

In [ ]:
random_policy = RandomPolicy(env.action_space.n)

record_simulation(
    gym.make("CartPole-v1", render_mode="rgb_array"),
    random_policy,
    "Video_CartPole_random_policy"
)

# Agent with REINFORCE: Normal vs Causality vs Simple Baseline vs Learned Baseline


### Policy Network Definition

To define a neural network that represents the agent’s policy for selecting actions.

The policy network takes the environment’s state as input and outputs a probability distribution over possible actions.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(PolicyNetwork, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, state):
        if not torch.is_tensor(state):
            state = torch.FloatTensor(state).to(DEVICE)
        
        logits = self.net(state)
        
        action_probs = torch.softmax(logits, dim=-1)
        
        return action_probs

### REINFORCE without Causality

#### 1. The Objective Function
In Policy Gradient methods, we aim to maximize the expected return:
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} [G(\tau)]$$
where $\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$ is a trajectory sampled from the policy $\pi_\theta$, and $G(\tau)$ is the total discounted return.

#### 2. The Policy Gradient Theorem
The gradient of the objective function is given by:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \left( \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t | s_t) \right) G(\tau) \right]$$

#### 3. Full Return vs. Reward-to-go
In the code implementation `reinforce_no_causality`, we use the **Full Return** $G_0$ for every action in the trajectory:
- **Definition of $G_0$**: $G_0 = \sum_{k=0}^{T} \gamma^k R_k$
- **Update Rule**: $\theta \leftarrow \theta + \alpha \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t | s_t) G_0$

#### Definitions:
- **$\pi_\theta(a_t | s_t)$**: The probability of taking action $a_t$ given state $s_t$ under parameters $\theta$.
- **$\nabla_\theta \log \pi_\theta(a_t | s_t)$**: The 'score function' or direction in parameter space that increases the probability of action $a_t$.
- **$G_0$**: The total scalar reward of the entire episode. If the episode was good, $G_0$ is high, and we push the probabilities of *all* actions taken in that episode 'up' equally.


In [ ]:
def reinforce_no_causality(
    env,
    policy_net,
    optimizer,
    num_episodes=1000,
    gamma=0.99,
    print_every=50,
    seed=333
):
    rewards_per_episode = []

    for episode in range(num_episodes):
        log_probs = []
        rewards = []

        state, info = env.reset(seed=seed + episode)
        done = False

        while not done:
            state_tensor = torch.FloatTensor(state).to(DEVICE)

            action_probs = policy_net(state_tensor)
            dist = Categorical(action_probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            next_state, reward, terminated, truncated, info = env.step(action.item())
            done = terminated or truncated

            rewards.append(reward)
            state = next_state

        G_0 = 0.0
        for t, reward in enumerate(rewards):
            G_0 += (gamma ** t) * reward

        policy_loss = []
        for log_prob in log_probs:
            policy_loss.append(-log_prob * G_0)

        policy_loss = torch.stack(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        episode_reward = float(sum(rewards))
        rewards_per_episode.append(episode_reward)

        if (episode + 1) % print_every == 0:
            print(f"[No Causality Trick] Episode {episode + 1}, Reward: {episode_reward}")

    return rewards_per_episode

**Question:**

Why does this method has high variance?



**Answer:**

Because it uses the full episode return to update the actions through the whole trajectory, even actions that happened before or had little effect. Also the full episode rewards can vary a lot.

### REINFORCE with Causality

To implement a function that calculates the discounted return for each timestep in an episode.

$$[
G_t = \sum_{k=0}^{T-t} \gamma^k R_{t+k}
]$$

In [ ]:
def compute_returns(rewards, gamma=0.99):
    returns = []
    G = 0

    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)

    return torch.FloatTensor(returns).to(DEVICE)

To train the agent using the standard policy gradient method.
The REINFORCE algorithm updates policy parameters by using the log-probability of actions multiplied by the discounted return.

This algorithm optimizes a **stochastic policy** $( \pi_{\theta}(a_t \mid s_t) )$ by updating its parameters in the direction that increases expected rewards. The update rule is based on the **policy gradient theorem**:  

$$[
\theta \leftarrow \theta + \alpha \sum_{t=0}^{T} \nabla_{\theta} \log \pi_{\theta}(a_t \mid s_t) G_t
]$$

where:  

- $( \theta )$ are the policy parameters (weights of the neural network).  
- $( \alpha )$ is the learning rate.  
- $( G_t )$ is the **discounted return** from timestep $( t )$:  

- $( \nabla_{\theta} \log \pi_{\theta}(a_t \mid s_t) )$ is the gradient of the log-probability of the selected action, used to adjust the policy in the correct direction.

In [ ]:
def reinforce(
    env,
    policy_net,
    optimizer,
    num_episodes=1000,
    gamma=0.99,
    print_every=50,
    seed=333
):
    rewards_per_episode = []

    for episode in range(num_episodes):
        log_probs = []
        rewards = []

        state, info = env.reset(seed=seed + episode)
        done = False

        while not done:
            state_tensor = torch.FloatTensor(state).to(DEVICE)

            action_probs = policy_net(state_tensor)
            dist = Categorical(action_probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))

            next_state, reward, terminated, truncated, info = env.step(action.item())
            done = terminated or truncated

            rewards.append(reward)
            state = next_state

        returns = compute_returns(rewards, gamma)

        policy_loss = []
        for log_prob, G_t in zip(log_probs, returns):
            policy_loss.append(-log_prob * G_t)

        policy_loss = torch.stack(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        episode_reward = float(sum(rewards))
        rewards_per_episode.append(episode_reward)

        if (episode + 1) % print_every == 0:
            print(f"[No Baseline] Episode {episode + 1}, Reward: {episode_reward}")

    return rewards_per_episode

### REINFORCE with a Constant Baseline

A simple way to reduce variance is to subtract a constant baseline from the returns. A common choice for this constant is the average return of the episode:

$$\theta \leftarrow \theta + \alpha \sum_{t=0}^{T} \nabla_{\theta} \log \pi_{\theta}(a_t | s_t) (G_t - b)$$

where $b = \frac{1}{T} \sum_{t=0}^T G_t$.

In [ ]:
def reinforce_constant_baseline(
    env,
    policy_net,
    optimizer,
    num_episodes=1000,
    gamma=0.99,
    print_every=50,
    seed=333
):
    rewards_per_episode = []

    for episode in range(num_episodes):
        log_probs = []
        rewards = []

        state, info = env.reset(seed=seed + episode)
        done = False

        while not done:
            state_tensor = torch.FloatTensor(state).to(DEVICE)

            action_probs = policy_net(state_tensor)
            dist = Categorical(action_probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))

            next_state, reward, terminated, truncated, info = env.step(action.item())
            done = terminated or truncated

            rewards.append(reward)
            state = next_state

        returns = compute_returns(rewards, gamma)

        baseline = returns.mean()
        advantages = returns - baseline

        policy_loss = []
        for log_prob, advantage in zip(log_probs, advantages):
            policy_loss.append(-log_prob * advantage)

        policy_loss = torch.stack(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        episode_reward = float(sum(rewards))
        rewards_per_episode.append(episode_reward)

        if (episode + 1) % print_every == 0:
            print(f"[Constant Baseline] Episode {episode + 1}, Reward: {episode_reward}")

    return rewards_per_episode

### Value Network Definition (Baseline Network)

Defines a neural network that approximates the value function, which estimates the expected return from a given state. The network takes the environment’s state as input and outputs a scalar value, representing the long-term reward the agent can expect from that state.

In [ ]:
class ValueNetwork(nn.Module):
    def __init__(self, input_dim):
        super(ValueNetwork, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, state):
        if not torch.is_tensor(state):
            state = torch.FloatTensor(state).to(DEVICE)

        return self.net(state)

### Implementing REINFORCE with a Learned Baseline

To reduce variance in policy updates, we introduce a **baseline** function $( V(s) )$, which estimates the expected return from a given state. Instead of using the full return $( G_t )$, we adjust the policy update by incorporating the baseline:  

$$[
\theta \leftarrow \theta + \alpha \sum_{t=0}^{T} \nabla_{\theta} \log \pi_{\theta}(a_t \mid s_t) (G_t - V(s_t))
]$$

where:  

- $( \theta )$ are the policy parameters (weights of the neural network).  
- $( \alpha )$ is the learning rate.  
- $( G_t )$ is the **discounted return** from timestep $( t )$:  

  $$[
  G_t = \sum_{k=0}^{T-t} \gamma^k R_{t+k}
  ]$$

- $( V(s_t) )$ is the **baseline function**, estimated by a **value network**, which is trained using mean squared error (MSE) between predicted values and actual returns:  

  $$[
  L = \frac{1}{N} \sum_{t=0}^{N} (G_t - V(s_t))^2
  ]$$

The introduction of the baseline does not change the expected value of the gradient but significantly **reduces variance**, leading to more stable learning.  


In [ ]:
def reinforce_with_baseline(
    env,
    policy_net,
    value_net,
    policy_optimizer,
    value_optimizer,
    num_episodes=1000,
    gamma=0.99,
    print_every=50,
    seed=333
):
    rewards_per_episode = []

    for episode in range(num_episodes):
        states = []
        log_probs = []
        rewards = []

        state, info = env.reset(seed=seed + episode)
        done = False

        while not done:
            state_tensor = torch.FloatTensor(state).to(DEVICE)

            action_probs = policy_net(state_tensor)
            dist = Categorical(action_probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            states.append(state)

            next_state, reward, terminated, truncated, info = env.step(action.item())
            done = terminated or truncated

            rewards.append(reward)
            state = next_state

        returns = compute_returns(rewards, gamma)
        states_tensor = torch.FloatTensor(np.array(states)).to(DEVICE)

        values = value_net(states_tensor).squeeze(-1)

        advantages = returns - values.detach()

        policy_loss = []
        for log_prob, advantage in zip(log_probs, advantages):
            policy_loss.append(-log_prob * advantage)

        policy_loss = torch.stack(policy_loss).sum()

        policy_optimizer.zero_grad()
        policy_loss.backward()
        policy_optimizer.step()

        value_loss = nn.MSELoss()(values, returns)

        value_optimizer.zero_grad()
        value_loss.backward()
        value_optimizer.step()

        episode_reward = float(sum(rewards))
        rewards_per_episode.append(episode_reward)

        if (episode + 1) % print_every == 0:
            print(f"[Learned Baseline] Episode {episode + 1}, Reward: {episode_reward}")

    return rewards_per_episode

### Training

To define key hyperparameters that influence training efficiency and convergence.

Hyperparameters like learning rate, discount factor, and number of episodes significantly impact the training process. Proper tuning is necessary to ensure stable learning while avoiding slow convergence or premature convergence to suboptimal policies.

In [ ]:
# TODO: Define all necessary hyperparameters

INPUT_DIM = env.observation_space.shape[0]
OUTPUT_DIM = env.action_space.n

LEARNING_RATE = 1e-3
DISCOUNT_FACTOR = 0.99
NUM_EPISODES = 2000

In [ ]:
# 1. Training REINFORCE without Causality (Full Return)
print("--- Training REINFORCE: No Causality (Full Return) ---")

set_seed(333)

train_env_no_causality = gym.make("CartPole-v1")
train_env_no_causality.action_space.seed(333)
train_env_no_causality.observation_space.seed(333)

policy_net_no_causality = PolicyNetwork(INPUT_DIM, OUTPUT_DIM).to(DEVICE)

optimizer_no_causality = optim.Adam(
    policy_net_no_causality.parameters(),
    lr=LEARNING_RATE
)

rewards_no_causality = reinforce_no_causality(
    train_env_no_causality,
    policy_net_no_causality,
    optimizer_no_causality,
    num_episodes=NUM_EPISODES,
    gamma=DISCOUNT_FACTOR,
    print_every=50,
    seed=333
)

train_env_no_causality.close()

In [ ]:
# 2. Training REINFORCE with Causality (No Baseline)
print("--- Training REINFORCE: With Causality (No Baseline) ---")

set_seed(333)

train_env_causality = gym.make("CartPole-v1")
train_env_causality.action_space.seed(333)
train_env_causality.observation_space.seed(333)

policy_net_causality = PolicyNetwork(INPUT_DIM, OUTPUT_DIM).to(DEVICE)

optimizer_causality = optim.Adam(
    policy_net_causality.parameters(),
    lr=LEARNING_RATE
)

rewards_no_baseline = reinforce(
    train_env_causality,
    policy_net_causality,
    optimizer_causality,
    num_episodes=NUM_EPISODES,
    gamma=DISCOUNT_FACTOR,
    print_every=50,
    seed=333
)

train_env_causality.close()

In [ ]:
# 3. Training REINFORCE with Constant Baseline
print("--- Training REINFORCE: Constant Baseline ---")

set_seed(333)

train_env_const_baseline = gym.make("CartPole-v1")
train_env_const_baseline.action_space.seed(333)
train_env_const_baseline.observation_space.seed(333)

policy_net_const_baseline = PolicyNetwork(INPUT_DIM, OUTPUT_DIM).to(DEVICE)

optimizer_const_baseline = optim.Adam(
    policy_net_const_baseline.parameters(),
    lr=LEARNING_RATE
)

rewards_constant_baseline = reinforce_constant_baseline(
    train_env_const_baseline,
    policy_net_const_baseline,
    optimizer_const_baseline,
    num_episodes=NUM_EPISODES,
    gamma=DISCOUNT_FACTOR,
    print_every=50,
    seed=333
)

train_env_const_baseline.close()

In [ ]:
# 4. Training REINFORCE with Learned Baseline
print("--- Training REINFORCE: Learned Baseline ---")

set_seed(333)

train_env_learned_baseline = gym.make("CartPole-v1")
train_env_learned_baseline.action_space.seed(333)
train_env_learned_baseline.observation_space.seed(333)

policy_net_learned_baseline = PolicyNetwork(INPUT_DIM, OUTPUT_DIM).to(DEVICE)
value_net_baseline = ValueNetwork(INPUT_DIM).to(DEVICE)

policy_opt_learned = optim.Adam(
    policy_net_learned_baseline.parameters(),
    lr=LEARNING_RATE
)

value_opt_learned = optim.Adam(
    value_net_baseline.parameters(),
    lr=LEARNING_RATE
)

rewards_with_baseline = reinforce_with_baseline(
    train_env_learned_baseline,
    policy_net_learned_baseline,
    value_net_baseline,
    policy_opt_learned,
    value_opt_learned,
    num_episodes=NUM_EPISODES,
    gamma=DISCOUNT_FACTOR,
    print_every=50,
    seed=333
)

train_env_learned_baseline.close()

# Plot and Comparing Results


In [ ]:
# TODO: Plotting Results
window_size = 50

def exponential_moving_average(data, window_size):
    # Convert to NumPy array
    data = np.array(data, dtype=np.float32)

    # EMA smoothing factor
    alpha = 2 / (window_size + 1)

    ema = np.zeros_like(data)
    ema[0] = data[0]

    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]

    return ema


# Plot the results for all 4 methods (Raw data + EMA)
methods = [
    (rewards_no_causality, "No Causality", "tab:red"),
    (rewards_no_baseline, "No Baseline", "tab:blue"),
    (rewards_constant_baseline, "Constant Baseline", "tab:orange"),
    (rewards_with_baseline, "Learned Baseline", "tab:green")
]

plt.figure(figsize=(14, 8))

for rewards, label, color in methods:
    episodes = np.arange(1, len(rewards) + 1)

    # Raw rewards
    plt.plot(
        episodes,
        rewards,
        color=color,
        alpha=0.25,
        linewidth=1,
        label=f"{label} Raw"
    )

    # EMA-smoothed rewards
    smoothed_rewards = exponential_moving_average(rewards, window_size)
    plt.plot(
        episodes,
        smoothed_rewards,
        color=color,
        linewidth=2.5,
        label=f"{label} EMA"
    )

plt.xlabel("Episode")
plt.ylabel("Episode Reward")
plt.title("REINFORCE Variants on CartPole-v1: Raw Rewards and EMA")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate and print statistics for all 4 methods
results_stats = [
    ("No Causality", rewards_no_causality),
    ("No Baseline", rewards_no_baseline),
    ("Constant Baseline", rewards_constant_baseline),
    ("Learned Baseline", rewards_with_baseline)
]

for name, rewards in results_stats:
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards)
    print(f"{name}: mean_reward = {mean_reward:.1f} +/- {std_reward:.1f}")

**Question:**

Based on the results, how does REINFORCE with a baseline compare to REINFORCE without a baseline in terms of performance?

**Answer:**

The one with baseline is better, in the results the no-baseline achieve 164.6 while the constant baseline is about 330. This shows adding baseline improves learning stability and overall performance by reducing the variance.


**Question:**

Based on the results, how does REINFORCE with a constant baseline compare to REINFORCE with a learned baseline in terms of performance?

**Answer:**

The constant one reward is about 330.5 while the learned one is about 425. This suggest that a learned value gives a more accurate estimate of the expected return from each state, as a result the variance is lower and the policy is improved.

**Question:**

Based on the results, how does REINFORCE without causality trick compare to REINFORCE with causality trick in terms of performance?


**Answer:**

Without that trick the reward is 38.7, but adding that trick moves us to 164. This shows that using reward to go instead of full episode return provide a better estimation for the gradient.

**Question:**

Explain how variance affects policy gradient methods, particularly in the context of estimating gradients from sampled trajectories.

**Answer:**

These methods estimate gradients using samples trajectories, so the gradient can have high variance, which means the update direction can vary a lot, even when the underlying policy hasn't changed much, so learning will be noisy, unstable and slower.

# Simulation

In [ ]:
# Record the simulation using the optimal policy no baseline
policy_net_causality.eval()

display(
    record_simulation(
        gym.make("CartPole-v1", render_mode="rgb_array"),
        policy_net_causality,
        "Video_CartPole_no_baseline"
    )
)

In [ ]:
# Record the simulation using the policy trained without causality trick
policy_net_no_causality.eval()

display(
    record_simulation(
        gym.make("CartPole-v1", render_mode="rgb_array"),
        policy_net_no_causality,
        "Video_CartPole_no_causality"
    )
)

In [ ]:
# Record the simulation using the policy trained with constant baseline
policy_net_const_baseline.eval()

display(
    record_simulation(
        gym.make("CartPole-v1", render_mode="rgb_array"),
        policy_net_const_baseline,
        "Video_CartPole_constant_baseline"
    )
)

In [ ]:
# Record the simulation using the optimal policy with baseline
policy_net_learned_baseline.eval()

display(
    record_simulation(
        gym.make("CartPole-v1", render_mode="rgb_array"),
        policy_net_learned_baseline,
        "Video_CartPole_learned_baseline"
    )
)

**Question:**

According to the simulations, how does each algorithm perform on the task after the training?

**Answer:**

The learned baseline is the best. The first two methods fail in the simulation test, but if we would say in order based on the previous results, we can say overall the performance order is: Learned Baseline > Constant Baseline > No Baseline > No Causality.
We know the learned baseline is the most stable one, The constant baseline is also performing well, the baselineless method is less stable and the first one is the noisiest one.